<center><h1><strong><font color="blue"> Data Engineering for Data Science - Week 07</font></strong></h1></center>

<center><img alt="" src="images/cover.png" style="height: 600px;"/></center>

<center><h2><strong><font color="blue">Case Study Answers</font></strong></h2></center>

<b><center><h3>(C) Taufik Sutanto</h3></center>

<center><h2><strong><font color="blue">Outline</font></strong></h2></center>

* **case Study: in-class Group Discussion**:
   - Normalizations
   - Basic Query
   - Join
   - Aggregations

<center><img alt="" src="images/class-exercise.png" style="height: 600px;"/></center>

# Please form groups consisting of three to four members.

<center><h1><strong><font color="blue"> Case Study: International Gym & Fitness Club </font></strong></h1></center>

## Background Information
The "Global Gym & Fitness Club" is a rapidly growing international fitness chain. To manage their operations, they currently use a single, large spreadsheet to track member registrations, class bookings, and instructor assignments. 

As the club expands to multiple countries, this "flat-file" approach has become a significant hurdle. The management is struggling with:
* **Data Entry Errors:** Inconsistent spelling of gym locations and instructor names.
* **Redundancy:** Every time a member joins a new class, their entire personal history is re-entered.
* **Inflexible Reporting:** It is difficult to calculate the total revenue per instructor or the popularity of specific class categories across different countries without complex manual filtering.

## The Learning Objective
In this exercise, students will act as Data Architects. Their task is to take this messy, unnormalized raw data and transform it into a structured, relational database. This transition will facilitate easier querying for business insights, such as:
1.  **Joins:** Connecting members to their specific classes and instructors.
2.  **Aggregations:** Calculating total members per country, average class prices, or total hours taught by each instructor.

<center><img alt="" src="images/deds-06/gym-db-casestudy.png" style="height: 600px;"/></center>

<center><h1><strong><font color="blue"> Case Study Data Sample: intl_gym_db </font></strong></h1></center>

## Download here: https://s.id/SCzm9

<center><h1><strong><font color="blue"> intl_gym_db: 1NF Solution? </font></strong></h1></center>

1. When developing the solution, how will you ensure that it is carried out efficiently?
2. Please exercise careful consideration in defining and using the “keys”.
3. You may include only a subset of the entries in your response.
4. As a guideline, please begin your table names with the prefix “1nf_” (for the purposes of this case study only).
5. Once you have developed your solution, ensure that it is properly imported into your database, taking care to avoid any potential table name conflicts.
6. The exercise should continue once at least one group has developed a solution and shared it with the rest of the class.
7. I will provide the solution after ten minutes; however, you are expected to formulate your own answer prior to that time.

In [4]:
# Solution here

# 1NF Answer:

* Primary Keys: registration_id, class_id, member_phone.
* Why not just registration_id and class_id?

## Download 1NF SQL script answer here: https://s.id/BUDCx

<center><img alt="" src="images/deds-06/gym-1NF.png" style="height: 480px;"/></center>

<center><h1><strong><font color="blue"> intl_gym_db: 2NF Solution? </font></strong></h1></center>

1. As a guideline, please begin your table names with the prefix “2nf_” (for the purposes of this case study only).
2. Please exercise careful consideration in defining and using the “keys”.
3. Your solution for the entries developed in the previous step should be consistently applied here.
4. The exercise should continue once at least one group has developed a solution and shared it with the rest of the class.
5. I will provide the solution after ten minutes; however, you are expected to formulate your own answer prior to that time.

In [5]:
# Solution here

# 2NF Answer:

1. **Registrations Table**: This captures the "Who, Where, and When" of the registration.
* **Primary Key:** `registration_id`
* **Columns:** `member_id`, `member_name`, `member_email`, `gym_location_id`, `gym_city`, `gym_country`, `booking_date`.

2. **Member Phones Table**: Since a member can have multiple phones (as seen in our 1NF split), we isolate this to prevent repeating the entire registration record for every phone number.
* **Primary Key:** `{registration_id, member_phone}`

3. **Classes Table**: This captures the "What" of the gym's offerings.
* **Primary Key:** `class_id`
* **Columns:** `class_name`, `class_type`, `instructor_name`, `class_fee`.

4. **Registration Details (The Link)**: This table connects registrations to classes. It answers: "Which classes did this registration include?"
* **Primary Key:** `{registration_id, class_id}`

## Download 2NF SQL script answer here: https://s.id/nHWFM
<center><img alt="" src="images/deds-06/gym-2NF.png" style="height: 480px;"/></center>

<center><h1><strong><font color="blue"> intl_gym_db: 3NF Solution? </font></strong></h1></center>

1. As a guideline, please begin your table names with the prefix “3nf_” (for the purposes of this case study only).
2. Please exercise careful consideration in defining and using the “keys”
3. Your solution for the entries developed in the previous step should be consistently applied here.
4. The exercise should continue once at least one group has developed a solution and shared it with the rest of the class.
5. I will provide the solution after ten minutes; however, you are expected to formulate your own answer prior to that time.

In [6]:
# Solution here

# 3NF Answer:

## Eliminating Transitive Dependencies
In our `registrations_2nf` table, we have two "middleman" dependencies where data is tied to the primary key (`registration_id`) through another non-key column:

1.  **The Member Dependency:** `registration_id` determines the `member_id`, but the `member_id` is what actually determines the `member_name` and `member_email`.
2.  **The Location Dependency:** `registration_id` determines the `gym_location_id`, but the `gym_location_id` is what determines the `gym_city` and `gym_country`.

## The Decomposition Process
We achieve 3NF by breaking these chains. We move the "middleman" data into its own dedicated table so that the information is stored exactly once.

### 1. Members Table
Isolates personal data from specific registration events.
* **Primary Key:** `member_id`
* **Columns:** `member_name`, `member_email`.

### 2. Locations Table
Isolates geographic data from the registrations.
* **Primary Key:** `gym_location_id`
* **Columns:** `gym_city`, `gym_country`.

### 3. Registrations Table (Refined)
Now this table only contains the "event" data and links to the other entities.
* **Primary Key:** `registration_id`
* **Foreign Keys:** `member_id`, `gym_location_id`.
* **Columns:** `booking_date`.

## Download 3NF SQL script answer here: https://s.id/JgSwF

<center><img alt="" src="images/deds-06/gym-3NF.png" style="height: 480px;"/></center>

In [2]:
# Python import the necessary modules
# Best Practice; use a function to connect to our database
import warnings; warnings.simplefilter('ignore')
import mysql.connector
import pandas as pd

host = "localhost" # modify as needed
user = "root" # modify as needed
password = "" # modify as needed
database = "intl_gym_db" # modify as needed

def connect(host="localhost", user="root", password="", database=""):
    try:
        db = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database)
        con = db.cursor()
        print("Connected to the '{}' Database.".format(database))
        return db, con
    except Exception as err_:
        print("Connection to the Database server failed:")
        print(err_)

<center><h1><strong><font color="blue"> intl_gym_db: Basic Query Exercise No 01 </font></strong></h1></center>

* Please use the final Third Normal Form (3NF) solution as the basis for this exercise.
* You may execute your queries using phpMyAdmin; the implementation in Python is left as an exercise.

## Basic Data Retrieval and Conditional Filtering
### Objective
The following exercise evaluates the student's proficiency in executing fundamental data retrieval operations on a single relation within the 3NF schema.

### Problem Statement
Identify and list the names of all gym classes categorized under 'Fitness' that possess a fee greater than 20.00.

In [1]:
db, con = connect(host=host, user=user, password=password, database=database)

# Your Answer here
query = """
....

"""


db, con = connect(host=host, user=user, password=password, database=database)

In [ ]:
query = """
-- Retrieve class names and fees based on specific criteria
SELECT 
    class_name, 
    class_fee
FROM classes_3nf
WHERE class_type = 'Fitness' 
  AND class_fee > 20.00;
"""

<center><h1><strong><font color="blue"> intl_gym_db: Basic Query Exercise No 02 </font></strong></h1></center>

# Exercise 2: Conditional Subqueries for Data Isolation

## Objective
This exercise evaluates the student's ability to utilize subqueries to filter data across distinct relations without employing explicit join syntax. 

## Problem Statement
Retrieve all `booking_date` entries from the registration records associated with the member named 'John Smith'.

## Logic
The query must first isolate the unique `member_id` from the `members_3nf` table and subsequently utilize that identifier to filter the relevant rows in the `registrations_3nf` table. This approach demonstrates how data from two tables can interact through nesting rather than merging.

## Expected Output
| booking_date |
| :--- |
| 2024-01-10 |
| 2024-02-01 |

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

# Your Answer here
query = """
....

"""


db, con = connect(host=host, user=user, password=password, database=database)

In [ ]:
query = """
SELECT 
    booking_date
FROM registrations_3nf
WHERE member_id = (
    SELECT member_id 
    FROM members_3nf 
    WHERE member_name = 'John Smith'
);
"""


<center><h1><strong><font color="blue"> intl_gym_db: Join Query Exercise No 01 </font></strong></h1></center>

# Exercise 3: Simple Two-Table Join

## Objective
Relate individual member identities to their specific transactional records using a primary-foreign key relationship.

## Problem Statement
List the `member_name` and their corresponding `booking_date` for all records.

## Expected Output
| member_name | booking_date |
| :--- | :--- |
| John Smith | 2024-01-10 |
| Anna Mueller | 2024-01-11 |
| Yuki Tanaka | 2024-01-12 |

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

# Your Answer here
query = """
....

"""


db, con = connect(host=host, user=user, password=password, database=database)

In [ ]:
query = """
SELECT 
    m.member_name, 
    r.booking_date
FROM members_3nf m
JOIN registrations_3nf r 
    ON m.member_id = r.member_id;
"""

<center><h1><strong><font color="blue"> intl_gym_db: Join Query Exercise No 02 </font></strong></h1></center>

# Exercise 4: Inner Join Practice

## Objective
Connect registration events to their specific gym locations by matching shared identifiers across two tables.

## Problem Statement
List every `registration_id` alongside its corresponding `gym_country`.

## Expected Output
| registration_id | gym_country |
| :--- | :--- |
| 1 | USA |
| 2 | Germany |
| 3 | Japan |

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

# Your Answer here
query = """
....

"""


db, con = connect(host=host, user=user, password=password, database=database)

In [ ]:
query = """
SELECT 
    r.registration_id, 
    l.gym_country
FROM registrations_3nf r
INNER JOIN locations_3nf l 
    ON r.gym_location_id = l.gym_location_id;
"""

<center><h1><strong><font color="blue"> intl_gym_db: Aggregation Query Exercise No 01 </font></strong></h1></center>

# Exercise 5: Basic Aggregation (AVG)

## Objective
Calculate a mean value from a numeric column using a single-table aggregate function to evaluate central tendencies within the dataset.

## Problem Statement
Calculate the average fee for all gym classes.

## Expected Output
| average_class_fee |
| :--- |
| 25.83 |

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

# Your Answer here
query = """
....

"""


db, con = connect(host=host, user=user, password=password, database=database)

In [ ]:
query = """
SELECT 
    AVG(class_fee) AS average_class_fee 
FROM classes_3nf;

"""

<center><h1><strong><font color="blue"> intl_gym_db: Aggregation Query Exercise No 02 </font></strong></h1></center>

# Exercise 6: Categorical Aggregation and Grouping

## Objective
This exercise evaluates the student's ability to consolidate records into logical categories using the `GROUP BY` clause and an aggregate function.

## Problem Statement
Determine the total number of classes available within each distinct `class_type`.

## Expected Output
| class_type | class_count |
| :--- | :--- |
| Fitness | 3 |
| Dance | 2 |
| Cardio | 1 |

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

# Your Answer here
query = """
....

"""


db, con = connect(host=host, user=user, password=password, database=database)

In [ ]:
query = """
SELECT 
    class_type, 
    COUNT(*) AS class_count
FROM classes_3nf
GROUP BY class_type;
"""

<center><h2><strong><font color="blue">End of Module</font></strong></h2></center>

<center><img alt="" src="images/meme-cartoon/meme-database-design.jpg" style="height: 600px;"/></center>